### cov file load

In [ ]:
import os

# 지정된 디렉토리 경로
directory1 = "./sample_data/step02_DMR/Normal/"
directory2 = "./sample_data/step02_DMR/GBM/"
types = '.sort.bismark.cov.gz'

def filename(dir, bedfile, type):
    for filename in os.listdir(dir):
        if filename.endswith(type):
            file_path = os.path.join(dir, filename)
            file_size = os.path.getsize(file_path)
            if file_size > 0:
                bedfile.append((filename, file_size))
    return bedfile

normal_samples = []
normal_samples = filename(directory1, normal_samples, types)
normal_samples = list(set(normal_samples))
normal_samples = [file[0] for file in sorted(normal_samples, key=lambda x: x[1], reverse=True)]
print(len(normal_samples))

cancer_samples = []
cancer_samples = filename(directory2, cancer_samples, types)
cancer_samples = list(set(cancer_samples))
cancer_samples = [file[0] for file in sorted(cancer_samples, key=lambda x: x[1], reverse=True)]
print(len(cancer_samples))

5
5


### 데이터 처리(100bp & 4 count 이상)

In [3]:
def group_bp(df, col, i, n):
    df[f'{col}_count{i}'] = df[f'{col}{i}'].notna().astype(int)  # Normal 열의 결측값 확인
    df[f'start_div_{n}'] = df['start'] // n
    df[f'end_div_{n}'] = df['end'] // n
    grouped = df.groupby(['chr', f'start_div_{n}'])[f'{col}_count{i}'].sum().reset_index()
    return grouped

In [4]:
import pandas as pd

n = 100

df_1 = pd.read_csv(rf'{directory1}{normal_samples[0]}', sep='\t', header=None, 
                   names=['chr', 'start', 'end', 'Normal1','tn','cn'], dtype={'chr': str})
df_1 = df_1[df_1.columns[:4]]
df_1['Normal_count1'] = df_1.iloc[:, 3:].notna().sum(axis=1)
df_1[f'start_div_{n}'] = df_1['start'] // n
df_1[f'end_div_{n}'] = df_1['end'] // n
grouped1 = df_1.groupby(['chr', f'start_div_{n}'])['Normal_count1'].sum().reset_index()

for i in range(2, len(normal_samples) + 1):
    df1_1 = pd.read_csv(rf'{directory1}{normal_samples[i - 1]}', sep='\t', header=None, 
                         names=['chr', 'start', 'end', f'Normal{i}','tn','cn'], dtype={'chr': str})
    df1_1 = df1_1[df1_1.columns[:4]]
    grouped1_1 = group_bp(df1_1, f'Normal', i, n)
    grouped1 = pd.merge(grouped1, grouped1_1, on=['chr', f'start_div_{n}'], how='inner')

grouped1

,chr,start_div_100,Normal_count1,Normal_count2,Normal_count3,Normal_count4,Normal_count5
0,1,104,1,1,1,1,1
1,1,105,6,6,6,6,6
2,1,156,1,1,1,1,1
3,1,157,1,1,1,1,1
4,1,161,2,2,2,2,2
...,...,...,...,...,...,...,...
1776566,Y,568798,1,1,2,2,3
1776567,Y,568799,7,7,7,7,8
1776568,Y,568801,5,5,5,5,2
1776569,Y,568819,2,2,1,1,1


In [5]:
df_2 = pd.read_csv(rf'{directory2}{cancer_samples[0]}', sep='\t', header=None, 
                   names=['chr', 'start', 'end', 'GBM1','tn','cn'], dtype={'chr': str})
df_2 = df_2[df_2.columns[:4]]
df_2['GBM_count1'] = df_2.iloc[:, 3:].notna().sum(axis=1)
df_2[f'start_div_{n}'] = df_2['start'] // n
df_2[f'end_div_{n}'] = df_2['end'] // n
grouped2 = df_2.groupby(['chr', f'start_div_{n}'])['GBM_count1'].sum().reset_index()

for i in range(2, len(cancer_samples) + 1):
    df2_1 = pd.read_csv(rf'{directory2}{cancer_samples[i - 1]}', sep='\t', header=None, 
                        names=['chr', 'start', 'end', f'GBM{i}','tn','cn'], dtype={'chr': str})
    df2_1 = df2_1[df2_1.columns[:4]]
    # print(df2_1)
    grouped2_1 = group_bp(df2_1, 'GBM', i, n)
    grouped2 = pd.merge(grouped2, grouped2_1, on=['chr', f'start_div_{n}'], how='inner')

grouped2

,chr,start_div_100,GBM_count1,GBM_count2,GBM_count3,GBM_count4,GBM_count5
0,1,104,1,1,1,1,1
1,1,105,6,6,6,6,6
2,1,515,2,2,2,2,2
3,1,516,9,9,9,9,9
4,1,517,1,1,1,1,1
...,...,...,...,...,...,...,...
545472,Y,568580,3,3,3,3,3
545473,Y,568581,3,3,3,5,3
545474,Y,568709,1,1,1,1,1
545475,Y,568798,1,1,1,1,1


In [6]:
df = pd.merge(grouped1, grouped2, on=['chr', f'start_div_{n}'], how='inner')
# df = df.drop(f'end_div_{n}', axis=1)
df

,chr,start_div_100,Normal_count1,Normal_count2,Normal_count3,Normal_count4,Normal_count5,GBM_count1,GBM_count2,GBM_count3,GBM_count4,GBM_count5
0,1,104,1,1,1,1,1,1,1,1,1,1
1,1,105,6,6,6,6,6,6,6,6,6,6
2,1,887,2,2,2,2,3,3,3,3,3,3
3,1,888,3,3,3,3,1,4,3,3,3,3
4,1,1363,1,1,1,1,2,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
399971,Y,568580,4,4,4,4,4,3,3,3,3,3
399972,Y,568581,5,5,5,5,8,3,3,3,5,3
399973,Y,568709,4,4,4,4,3,1,1,1,1,1
399974,Y,568798,1,1,2,2,3,1,1,1,1,1


In [ ]:
df.to_csv(f'./results/step02_DMR/sample_div_{n}_v1.csv', index=False)

In [9]:
th=4

df2 = df[(df.iloc[:, 2:] >= th).all(axis=1)]
df2

,chr,start_div_100,Normal_count1,Normal_count2,Normal_count3,Normal_count4,Normal_count5,GBM_count1,GBM_count2,GBM_count3,GBM_count4,GBM_count5
1,1,105,6,6,6,6,6,6,6,6,6,6
5,1,1364,10,10,5,5,10,5,9,9,4,9
19,1,7780,9,9,5,5,9,5,5,5,5,5
25,1,8277,11,11,11,11,11,11,11,11,11,11
32,1,8699,14,14,9,9,5,9,9,12,9,9
...,...,...,...,...,...,...,...,...,...,...,...,...
399960,Y,113335,8,8,9,9,8,12,11,11,11,11
399964,Y,113340,4,4,4,4,23,15,24,14,15,14
399968,Y,114555,7,7,7,7,7,7,7,7,7,7
399970,Y,129006,6,6,11,11,10,12,9,4,5,9


In [ ]:
df2.to_csv(f'./results/step02_DMR/{n}기준_{th}이상_v1.csv', index=False)

In [11]:
def mean_bp(df, col, i, n, g_list):
    df[f'start_div_{n}'] = df['start'] // n
    grouped = df.groupby(['chr', f'start_div_{n}'])[f'{col}{i}'].mean().reset_index()
    grouped_df = pd.merge(g_list, grouped, on=['chr', f'start_div_{n}'], how='inner')
    return grouped_df

In [ ]:
import pandas as pd

n=100
th=4

df2 = pd.read_csv(f'./results/step02_DMR/{n}기준_{th}이상_v1.csv')
g_list = df2[['chr','start_div_100']]

df_1 = pd.read_csv(rf'{directory1}{normal_samples[0]}', sep='\t', header=None, names=['chr', 'start', 'end', 'Normal1','tn','cn'], dtype={'chr': str})
df_1 = df_1[df_1.columns[:4]]
df_1[f'start_div_{n}'] = df_1['start'] // n
grouped1_2 = df_1.groupby(['chr', f'start_div_{n}'])['Normal1'].mean().reset_index()
grouped1_2 = pd.merge(g_list, grouped1_2, on=['chr', f'start_div_{n}'], how='inner')

for i in range(2, len(normal_samples)+1):
    df1_1 = pd.read_csv(rf'{directory1}{normal_samples[i - 1]}', sep='\t', header=None, 
                        names=['chr', 'start', 'end', f'Normal{i}','tn','cn'], dtype={'chr': str})
    df1_1 = df1_1[df1_1.columns[:4]]
    grouped1_3 = mean_bp(df1_1, 'Normal', i, n, g_list)
    grouped1_2 = pd.merge(grouped1_2, grouped1_3, on=['chr', f'start_div_{n}'], how='inner')

grouped1_2

,chr,start_div_100,Normal1,Normal2,Normal3,Normal4,Normal5
0,8,262913,0.000000,0.000000,0.000000,0.000000,1.250000
1,8,263345,100.000000,100.000000,100.000000,100.000000,100.000000
2,8,263556,93.750000,93.750000,100.000000,100.000000,95.833333
3,8,263694,100.000000,100.000000,100.000000,100.000000,100.000000
4,8,263831,0.000000,0.000000,3.703704,3.703704,0.000000
...,...,...,...,...,...,...,...
14711,Y,113335,54.864130,54.864130,51.747635,51.747635,54.714782
14712,Y,113340,92.857143,92.857143,85.000000,85.000000,88.260870
14713,Y,114555,52.380952,52.380952,64.285714,64.285714,54.761905
14714,Y,129006,50.000000,50.000000,72.727273,72.727273,90.000000


In [ ]:
import pandas as pd

n=100
th=4

df2 = pd.read_csv(f'./results/step02_DMR/{n}기준_{th}이상_v1.csv')
g_list = df2[['chr','start_div_100']]

df_2 = pd.read_csv(rf'{directory2}{cancer_samples[0]}', sep='\t', header=None, names=['chr', 'start', 'end', 'GBM1','tn','cn'], dtype={'chr': str})
df_2 = df_2[df_2.columns[:4]]
df_2[f'start_div_{n}'] = df_2['start'] // n
grouped2_2 = df_2.groupby(['chr', f'start_div_{n}'])['GBM1'].mean().reset_index()
grouped2_2 = pd.merge(g_list, grouped2_2, on=['chr', f'start_div_{n}'], how='inner')

for i in range(2, len(cancer_samples)+1):
    df2_1 = pd.read_csv(rf'{directory2}{cancer_samples[i - 1]}', sep='\t', header=None, names=['chr', 'start', 'end', f'GBM{i}','tn','cn'], dtype={'chr': str})
    df2_1 = df2_1[df2_1.columns[:4]]
    grouped2_3 = mean_bp(df2_1, 'GBM', i, n, g_list)
    grouped2_2 = pd.merge(grouped2_2, grouped2_3, on=['chr', f'start_div_{n}'], how='inner')

grouped2_2

,chr,start_div_100,GBM1,GBM2,GBM3,GBM4,GBM5
0,8,262913,0.714286,0.729301,0.000000,1.428571,2.379230
1,8,263345,97.559524,80.000000,100.000000,98.529412,100.000000
2,8,263556,100.000000,96.428571,98.809524,100.000000,100.000000
3,8,263694,97.872340,97.712418,89.761905,97.272727,93.846154
4,8,263831,0.877340,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...
14711,Y,113335,15.537371,26.214226,24.196397,14.751671,23.481758
14712,Y,113340,59.881181,77.100885,59.334761,58.361630,62.240336
14713,Y,114555,1.465201,3.726708,2.316602,1.503759,0.269542
14714,Y,129006,97.222222,100.000000,100.000000,100.000000,96.296296


In [14]:
df3 = pd.merge(grouped1_2, grouped2_2, on=['chr', f'start_div_{n}'], how='inner')
df3.iloc[:, 2:] = df3.iloc[:, 2:] / 100
df3['chr_start'] = df3['chr'].astype(str) + '_' + df3[f'start_div_{n}'].astype(str)
df3 = df3.drop(['chr',f'start_div_{n}'], axis=1)
df3 = df3.set_index('chr_start').transpose()
df3.loc[df3.index[:10], 'Type'] = 0
df3.loc[df3.index[10:], 'Type'] = 1

df3

chr_start,8_262913,8_263345,8_263556,8_263694,8_263831,8_263835,8_264340,8_264341,8_264412,8_264484,...,Y_113325,Y_113326,Y_113333,Y_113334,Y_113335,Y_113340,Y_114555,Y_129006,Y_568799,Type
Normal1,0.000000,1.000000,0.937500,1.000000,0.000000,0.0,1.000000,0.750000,0.000000,0.025000,...,0.692857,0.507003,0.800000,0.547222,0.548641,0.928571,0.523810,0.500000,0.907143,0.0
Normal2,0.000000,1.000000,0.937500,1.000000,0.000000,0.0,1.000000,0.750000,0.000000,0.025000,...,0.692857,0.507003,0.800000,0.547222,0.548641,0.928571,0.523810,0.500000,0.907143,0.0
Normal3,0.000000,1.000000,1.000000,1.000000,0.037037,0.0,0.952381,0.750000,0.000000,0.000000,...,0.533333,0.581232,0.420833,0.366305,0.517476,0.850000,0.642857,0.727273,0.714286,0.0
Normal4,0.000000,1.000000,1.000000,1.000000,0.037037,0.0,0.952381,0.750000,0.000000,0.000000,...,0.533333,0.581232,0.420833,0.366305,0.517476,0.850000,0.642857,0.727273,0.714286,0.0
Normal5,0.012500,1.000000,0.958333,1.000000,0.000000,0.0,1.000000,0.916667,0.000000,0.000000,...,0.729167,0.637255,0.666667,0.423481,0.547148,0.882609,0.547619,0.900000,0.725000,0.0
GBM1,0.007143,0.975595,1.000000,0.978723,0.008773,0.0,0.956522,0.931677,0.804772,0.006938,...,0.443034,0.215217,0.036302,0.126162,0.155374,0.598812,0.014652,0.972222,0.007356,0.0
GBM2,0.007293,0.800000,0.964286,0.977124,0.000000,0.0,0.983871,0.967742,0.775481,0.008065,...,0.608396,0.458466,0.018712,0.099666,0.262142,0.771009,0.037267,1.000000,0.003140,0.0
GBM3,0.000000,1.000000,0.988095,0.897619,0.000000,0.0,0.928571,0.990000,0.677040,0.003125,...,0.020000,0.426808,0.057701,0.097246,0.241964,0.593348,0.023166,1.000000,0.003968,0.0
GBM4,0.014286,0.985294,1.000000,0.972727,0.000000,0.0,0.875000,0.928571,0.828655,0.048699,...,0.187683,0.411439,0.047999,0.105776,0.147517,0.583616,0.015038,1.000000,0.009368,0.0
GBM5,0.023792,1.000000,1.000000,0.938462,0.000000,0.0,0.943676,0.965217,0.747135,0.000933,...,0.387500,0.317513,0.029032,0.096599,0.234818,0.622403,0.002695,0.962963,0.015189,0.0


In [ ]:
df3.to_csv(f'./results/step02_DMR/{n}기준_{th}이상_평균_v1.csv', index=False)

### 처리한 data load

In [ ]:
import pandas as pd

n=100
th=4

df3 =pd.read_csv(f'./results/step02_DMR/{n}기준_{th}이상_평균_v1.csv')
df3

,1_105,1_8274,1_8700,1_9102,1_9103,1_9206,1_9237,1_9246,1_9405,1_9597,...,X_1558813,Y_113065,Y_113075,Y_113148,Y_113215,Y_113325,Y_113326,Y_113334,Y_113335,Type
0,1.000000,0.000000,0.033333,0.015942,0.000000,0.154762,0.000000,0.026087,0.009615,0.605263,...,0.166667,0.900000,0.645455,0.857143,0.850000,0.692857,0.507003,0.547222,0.548641,0.0
1,0.857143,0.000000,0.000000,0.058824,0.000000,0.384615,0.000000,0.000000,0.011765,0.303030,...,0.500000,0.800000,1.000000,0.772222,0.600000,0.480000,0.595098,0.439205,0.588848,0.0
2,0.833333,0.000000,0.000000,0.000000,0.041667,0.384615,0.000000,0.000000,0.000000,0.336538,...,0.483135,0.812500,0.960000,0.897959,0.750000,0.750000,0.592157,0.554985,0.461310,0.0
3,0.861111,0.000000,0.000000,0.000000,0.000000,0.428571,0.000000,0.017391,0.000000,0.600000,...,0.361111,0.937500,0.600000,0.500000,0.733333,0.533333,0.581232,0.366305,0.517476,0.0
4,0.785714,0.047619,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,...,0.555556,0.785985,0.933333,0.750000,0.800000,0.562500,0.666667,0.602306,0.502723,0.0
5,0.857143,0.000000,0.000000,0.058824,0.000000,0.384615,0.000000,0.000000,0.011765,0.303030,...,0.500000,0.800000,1.000000,0.772222,0.600000,0.480000,0.595098,0.439205,0.588848,0.0
6,1.000000,0.000000,0.000000,0.022727,0.033333,0.062500,0.016667,0.000000,0.000000,0.250000,...,0.000000,0.816319,0.716667,0.893750,0.771429,0.650000,0.567647,0.478838,0.643275,0.0
7,1.000000,0.000000,0.083333,0.000000,0.000000,0.142857,0.000000,0.000000,0.000000,0.666667,...,0.666667,0.700000,0.600000,0.638889,0.800000,0.640000,0.616667,0.392232,0.505847,0.0
8,0.666667,0.071429,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,...,0.428571,0.621528,0.791667,0.875000,0.777778,0.812500,0.706250,0.485714,0.513393,0.0
9,0.833333,0.000000,0.000000,0.000000,0.041667,0.384615,0.000000,0.000000,0.000000,0.336538,...,0.483135,0.812500,0.960000,0.897959,0.750000,0.750000,0.592157,0.554985,0.461310,0.0


### RandomForest

In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

X = df3[df3.columns[:-1]]

# X = X.drop(columns=['Unnamed: 0'])
y = df3['Type']

rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_

indices = np.argsort(importances)[::-1]
RandomForest_df = pd.DataFrame({
    'Feature': X.columns[indices],
    'RandomForest': importances[indices].round(4)
})

RandomForest_df = RandomForest_df[RandomForest_df['RandomForest'] > 0]

RandomForest_df = RandomForest_df.head(100)
RandomForest_df

,Feature,RandomForest
0,7_484545,0.01
1,2_915882,0.01
2,20_297428,0.01
3,20_616282,0.01
4,9_1076375,0.01
...,...,...
95,1_1570898,0.01
96,1_1551437,0.01
97,1_323623,0.01
98,1_24159,0.01


### XGB

In [4]:
import xgboost as xgb

X = df3[df3.columns[:-1]]
# X = X.drop(columns=['Unnamed: 0'])
y = df3['Type']

# XGBoost 모델 훈련
model = xgb.XGBClassifier(random_state=42)
model.fit(X, y)

# Feature importances 얻기
importances = model.feature_importances_

# 인덱스를 중요도에 따라 정렬
indices = np.argsort(importances)[::-1]
XGB_df = pd.DataFrame({
    'Feature': X.columns[indices],
    'XGBoost': importances[indices].round(4)
})

XGB_df = XGB_df[XGB_df['XGBoost'] > 0]

XGB_df = XGB_df[:100]
XGB_df

,Feature,XGBoost
0,1_9597,1.0


### LogisticRegression

In [5]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression 모델 훈련
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

# Feature importances (계수) 얻기
importances = model.coef_[0]
indices = np.argsort(np.abs(importances))[::-1]  # 절댓값을 기준으로 정렬

# Feature 이름과 중요도 출력
print("Feature importances:")
LogisticRegression_df = pd.DataFrame({
    'Feature': X.columns[indices],
    'Logistic_Regression': importances[indices].round(4)
})

LogisticRegression_df = LogisticRegression_df[:100]
LogisticRegression_df

Feature importances:


,Feature,Logistic_Regression
0,6_415482,0.0068
1,8_1410362,0.0068
2,18_502683,0.0068
3,16_112332,0.0068
4,5_935708,0.0068
...,...,...
95,5_1785901,0.0065
96,14_522679,0.0065
97,8_389745,0.0065
98,12_540297,0.0065


### T-Test

In [6]:
from scipy import stats

t_test_results = {}
X = df3[df3.columns[:-1]]
# X = X.drop(columns=['Unnamed: 0'])
y = df3['Type']
type_0 = X[y == 0]
type_1 = X[y == 1]

for column in X.columns:
    t_stat, p_val = stats.ttest_ind(type_0[column], type_1[column], equal_var=False)
    t_test_results[column] = p_val

t_test_df = pd.DataFrame({
    'Feature': list(t_test_results.keys()),
    'p_value': list(t_test_results.values())
}).sort_values(by='p_value')

## 조건
T_test__df = t_test_df[t_test_df['p_value'] < 0.05]
T_test__df = T_test__df.head(100)
T_test__df

,Feature,p_value
6497,14_1043744,1.208133e-29
7230,15_893790,8.739243e-29
6754,15_414954,1.016173e-28
20468,7_1162104,6.767609e-28
13387,2_2194486,6.958898e-28
...,...,...
22976,9_1377869,7.343076e-20
3486,11_318052,7.505917e-20
21226,8_316407,7.648074e-20
14886,22_197242,8.194780e-20


### LogisticRegression & T-test 공통 중요 피처 확인

In [7]:
common_features = pd.merge(LogisticRegression_df,T_test__df,on='Feature',how='inner')
# feature_importances_df = pd.merge(feature_importances_df,feature_importances_df3,on='Feature',how='inner')
# feature_importances_df = feature_importances_df.sort_values(by='Feature')
common_features


print("Logistic Regression Importance for All Features:\n", LogisticRegression_df)
print("\nT-test p-values for All Features:\n", T_test__df)
print("\nCommon Features (Importance > 0 & p-value < 0.05) with Importance and p-value:\n", common_features)

Logistic Regression Importance for All Features:
       Feature  Logistic_Regression
0    6_415482               0.0068
1   8_1410362               0.0068
2   18_502683               0.0068
3   16_112332               0.0068
4    5_935708               0.0068
..        ...                  ...
95  5_1785901               0.0065
96  14_522679               0.0065
97   8_389745               0.0065
98  12_540297               0.0065
99  14_746119               0.0065

[100 rows x 2 columns]

T-test p-values for All Features:
           Feature       p_value
6497   14_1043744  1.208133e-29
7230    15_893790  8.739243e-29
6754    15_414954  1.016173e-28
20468   7_1162104  6.767609e-28
13387   2_2194486  6.958898e-28
...           ...           ...
22976   9_1377869  7.343076e-20
3486    11_318052  7.505917e-20
21226    8_316407  7.648074e-20
14886   22_197242  8.194780e-20
14652   21_423214  1.017237e-19

[100 rows x 2 columns]

Common Features (Importance > 0 & p-value < 0.05) with Import

In [ ]:
common_features.to_csv(f'./results/step02_DMR/{n}기준_{th}이상_feature_importances_v2(inner).csv', index=False)